In [3]:
import pandas as pd
import os

DATA_PATH = r"C:\Personal Projects\MMLM-2026\data\raw"

seeds = pd.read_csv(f"{DATA_PATH}/MNCAATourneySeeds.csv")
tourney  = pd.read_csv(f"{DATA_PATH}/MNCAATourneyCompactResults.csv")
detailed = pd.read_csv(f"{DATA_PATH}/MRegularSeasonDetailedResults.csv")
massey   = pd.read_csv(f"{DATA_PATH}/MMasseyOrdinals.csv")
sample   = pd.read_csv(f"{DATA_PATH}/SampleSubmissionStage2.csv")

print("=== Seeds ===")
print(seeds.shape)
print(seeds.head(3).to_string())
print("\n=== Tourney Results ===")
print(tourney.shape)
print(tourney.head(3).to_string())
print("\n=== Detailed Results ===")
print(detailed.shape)
print(detailed.head(3).to_string())
print("\n=== Massey Ordinals ===")
print(massey.shape, massey['SystemName'].nunique(), "systems")
print(massey.head(3).to_string())
print("\n=== Sample Submission (Stage 2) ===")
print(sample.shape)
print(sample.head(5).to_string())

=== Seeds ===
(2626, 3)
   Season Seed  TeamID
0    1985  W01    1207
1    1985  W02    1210
2    1985  W03    1228

=== Tourney Results ===
(2585, 8)
   Season  DayNum  WTeamID  WScore  LTeamID  LScore WLoc  NumOT
0    1985     136     1116      63     1234      54    N      0
1    1985     136     1120      59     1345      58    N      0
2    1985     136     1207      68     1250      43    N      0

=== Detailed Results ===
(122775, 34)
   Season  DayNum  WTeamID  WScore  LTeamID  LScore WLoc  NumOT  WFGM  WFGA  WFGM3  WFGA3  WFTM  WFTA  WOR  WDR  WAst  WTO  WStl  WBlk  WPF  LFGM  LFGA  LFGM3  LFGA3  LFTM  LFTA  LOR  LDR  LAst  LTO  LStl  LBlk  LPF
0    2003      10     1104      68     1328      62    N      0    27    58      3     14    11    18   14   24    13   23     7     1   22    22    53      2     10    16    22   10   22     8   18     9     2   20
1    2003      10     1272      70     1393      63    N      0    26    62      8     20    10    19   15   28    16   13

In [ ]:
import pandas as pd
import numpy as np

DATA_PATH = r"C:\Personal Projects\MMLM-2026\data\raw"

detailed = pd.read_csv(f"{DATA_PATH}/MRegularSeasonDetailedResults.csv")
compact  = pd.read_csv(f"{DATA_PATH}/MRegularSeasonCompactResults.csv")
seeds    = pd.read_csv(f"{DATA_PATH}/MNCAATourneySeeds.csv")
tourney  = pd.read_csv(f"{DATA_PATH}/MNCAATourneyCompactResults.csv")
massey   = pd.read_csv(f"{DATA_PATH}/MMasseyOrdinals.csv")
coaches  = pd.read_csv(f"{DATA_PATH}/MTeamCoaches.csv")
teams    = pd.read_csv(f"{DATA_PATH}/MTeams.csv")

# Sanity checks
print("Detailed seasons:", detailed['Season'].min(), "–", detailed['Season'].max())
print("Compact seasons:", compact['Season'].min(), "–", compact['Season'].max())
print("Tourney seasons:", tourney['Season'].min(), "–", tourney['Season'].max())
print("Seeds seasons:", seeds['Season'].min(), "–", seeds['Season'].max())
print("Massey systems sample:", massey['SystemName'].unique()[:20])
print("Coaches sample:\n", coaches.head(5).to_string())
print("\nAny nulls in detailed?", detailed.isnull().sum().sum())
print("Any nulls in compact?", compact.isnull().sum().sum())

Detailed seasons: 2003 – 2026
Compact seasons: 1985 – 2026
Tourney seasons: 1985 – 2025
Seeds seasons: 1985 – 2026
Massey systems sample: ['SEL' 'AP' 'BIH' 'DUN' 'ENT' 'GRN' 'IMS' 'MAS' 'MKV' 'MOR' 'POM' 'RPI'
 'SAG' 'SAU' 'SE' 'STR' 'USA' 'WLK' 'WOB' 'BOB']
Coaches sample:
    Season  TeamID  FirstDayNum  LastDayNum       CoachName
0    1985    1102            0         154   reggie_minton
1    1985    1103            0         154     bob_huggins
2    1985    1104            0         154  wimp_sanderson
3    1985    1106            0         154    james_oliver
4    1985    1108            0         154   davey_whitney

Any nulls in detailed? 0
Any nulls in compact? 0


In [ ]:
import pandas as pd

DATA_PATH = r"C:\Personal Projects\MMLM-2026\data\raw"

massey   = pd.read_csv(f"{DATA_PATH}/MMasseyOrdinals.csv")
teams    = pd.read_csv(f"{DATA_PATH}/MTeams.csv")

# Check coverage of Massey systems by season
for system in ['POM', 'SAG', 'MAS']:
    coverage = (massey[massey['SystemName'] == system]
                .groupby('Season')['TeamID'].count())
    print(f"\n{system} coverage:")
    print(f"  Seasons: {coverage.index.min()} - {coverage.index.max()}")
    print(f"  2024 teams: {coverage.get(2024, 0)}")
    print(f"  2025 teams: {coverage.get(2025, 0)}")
    print(f"  2026 teams: {coverage.get(2026, 0)}")

# Find systems with full coverage through 2026
system_coverage = (massey.groupby('SystemName')['Season']
                   .max()
                   .reset_index()
                   .rename(columns={'Season':'MaxSeason'}))

# Filter to systems that cover 2026 and have been around since at least 2010
system_start = (massey.groupby('SystemName')['Season']
                .min()
                .reset_index()
                .rename(columns={'Season':'MinSeason'}))

coverage_full = (system_coverage
                 .merge(system_start, on='SystemName')
                 .query('MaxSeason == 2026 and MinSeason <= 2010')
                 .sort_values('MinSeason'))

print(f"Systems with coverage 2010-2026: {len(coverage_full)}")
print(coverage_full.to_string(index=False))


# Check NET coverage in the Massey data
net_check = massey[massey['SystemName'].str.contains('NET', case=False, na=False)]
print("NET-related systems:", net_check['SystemName'].unique())
print("NET coverage by season:")
if len(net_check) > 0:
    print(net_check.groupby(['SystemName','Season'])['TeamID'].count().reset_index().to_string())


POM coverage:
  Seasons: 2003 - 2026
  2024 teams: 6878
  2025 teams: 6916
  2026 teams: 7300

SAG coverage:
  Seasons: 2003 - 2023
  2024 teams: 0
  2025 teams: 0
  2026 teams: 0

MAS coverage:
  Seasons: 2003 - 2026
  2024 teams: 6878
  2025 teams: 6916
  2026 teams: 7300
Systems with coverage 2010-2026: 24
SystemName  MaxSeason  MinSeason
        AP       2026       2003
       USA       2026       2003
       RTH       2026       2003
       RPI       2026       2003
       POM       2026       2003
       MOR       2026       2003
       WLK       2026       2003
       MAS       2026       2003
       WOL       2026       2003
       DUN       2026       2003
       DOL       2026       2003
       COL       2026       2003
       BIH       2026       2003
       DES       2026       2004
       WIL       2026       2005
       DOK       2026       2006
       KPK       2026       2006
        MB       2026       2006
       PGH       2026       2008
       REW       2026       

In [ ]:
# Team lookup!

DATA_PATH = r"C:\Personal Projects\MMLM-2026\data\raw"

teams    = pd.read_csv(f"{DATA_PATH}/MTeams.csv")

opp_ids = [1181, 1196, ]
print(teams[teams['TeamID'].isin(opp_ids)][['TeamID','TeamName']])

     TeamID       TeamName
56     1157    Coastal Car
195    1296     N Illinois
278    1379  Southern Miss
318    1419            ULM
